# Omni ChromHMM Analysis — ENCODE

Analysis, cross-segmentation comparison and inter-dataset summary plots of the ENCODE
datasets `config_encode.yaml` names.

Run the Snakemake pipeline first to produce the segmentations (`{ds}/.done`), then this
notebook top-to-bottom. **Setup** loads the configuration and defines the helpers every
section below shares; each numbered section then runs the same three steps:

1. *Compute* — analysis and comparison, cached on disk, so a re-run only fills gaps
2. *Plotting* — every figure written under `out/`, nothing rendered
3. *Display* — the figures shown inline, missing ones skipped silently

The sections are:

1. **Segmentation analysis and cross-dataset summaries** — the per-segmentation
   analysis, the per-dataset and inter-dataset comparisons, and the summary plots:
   peaks, segment counts, state lengths, composition, entropy, biological validation
2. **Optimal number of states** — inertia, silhouette and transition entropy over a
   sweep of the requested state count
3. **Cell type differences in chromatin** — how often the same locus keeps its state
   across the datasets, per method
4. **Joint models across replicates** — two models learned per replicate against one
   model learned over both
5. **Sample-pair agreement** — the same method on two different samples: across
   samples (same assay) and across assays (ChIP-seq vs Mint-ChIP)
6. **Functional validation** — the state families against the sample's own ATAC-seq,
   expressed TSS and expressed gene bodies

Sections 4-6 write the caches `summary.ipynb` reads (`out/df_joint_rep.pkl`,
`out/df_joint_indiv.pkl`, `out/df_cross_sample.pkl`, `out/df_cross_assay.pkl`).

## Setup

In [ ]:
import glob
import importlib
import multiprocessing as mp
import os
import sys
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

In [ ]:
# Load configuration
config_path = os.path.abspath("config_encode.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

project_root = os.path.dirname(config_path)
scripts_dir = os.path.join(project_root, "scripts", "analysis")
scripts_rules_dir = os.path.join(project_root, "scripts", "rules")
scripts_root_dir = os.path.join(project_root, "scripts")
workdir = os.path.expanduser(config.get("workdir", "."))

In [ ]:
# Import the analysis methods directly (no CLI / subprocess).
sys.path.insert(0, scripts_dir)
sys.path.insert(0, scripts_rules_dir)
sys.path.insert(0, scripts_root_dir)

import analyze
import analyze_peaks
import compare
import compare_methods
import compare_inter_dataset as compare_out
import display as _display_helpers
import emission_similarity
import match
import summary_plots
import utils
from utils import (
    CHROMHMM, CHROMHMM_DEFAULT, CHROMHMM_DISPLAY, COSINE, COSINE_DISPLAY,
    F1_DISPLAY, FULL, FULL_DISPLAY, HOMER, HOMER_DISPLAY, JACCARD,
    JACCARD_DISPLAY, JOINT_CHROMHMM, KAPPA, KAPPA_DISPLAY, MACS2,
    MACS2_DISPLAY, NOQH, NOQH_DISPLAY, NOQH_STATES, OMNI, OMNI_DISPLAY
)

# Re-import the analysis modules so edits to scripts/analysis/*.py are picked up when this
# cell is re-run, without needing a kernel restart (plain `import` caches modules).
for _m in (analyze, analyze_peaks, compare, compare_methods,
           compare_out, emission_similarity, match,
           summary_plots, utils, _display_helpers):
    importlib.reload(_m)

# Display helpers, shared with the other analysis notebooks: every plot the
# Display cells show was produced by a Plotting cell and is read straight off
# disk, and a missing file is skipped, so the notebook renders whatever the
# pipeline produced. Imported as functions, not as the module: `display` is
# also the name of IPython's own display().
from display import header, method_plot, show_group, show_table

# Run everything relative to the pipeline working directory.
os.chdir(workdir)
print(f"Project root: {project_root}")
print(f"Scripts dir : {scripts_dir}")
print(f"Working dir : {workdir}")

In [ ]:
# Parameters, mirroring the Snakefile.
P = config["params"]
TOOLS = config["tools"]
DATASETS = config["datasets"]
OUTLIERS = ["uterus", "thyroid_gland"]
# Don't process outliers in the analysis.
DATASETS = {k: v for k, v in DATASETS.items() if k not in OUTLIERS}
MARKS = ["H3K36me3", "H3K9me3", "H3K4me1", "H3K27ac", "H3K27me3", "H3K4me3"]
CHROMHMM_BIN = P["chromhmm_bin"]
NSTATES = P["n_states"]
DO_REPLICATES = P.get("replicates", False)
# The match variant every comparison below is read off: the segmentations
# relabelled to their dataset's ENCODE reference markup.
MATCH_METHOD = "matched"

# Peak callers to include. Edit to match the segmentations you actually produced;
# missing files are skipped gracefully throughout the notebook.
CALLERS = [HOMER, MACS2, OMNI]
CALLER_BIN = {OMNI: P["omni_bin"], HOMER: P["homer_bin"], MACS2: P["macs2_bin"]}

COORDS_DIR = os.path.join(workdir, TOOLS["coords_dir"])
GENCODE_GTF = os.path.join(workdir, TOOLS["gencode_gtf"])

# De-novo methods compared across datasets.
INTER_DS_METHODS = [CHROMHMM_DEFAULT] + [utils.method_key(c) for c in CALLERS]
# Joint models: one model per dataset over both of its replicates, see
# process_encode.sh. Same order as INTER_DS_METHODS, so the two lists pair up.
JOINT_METHODS = [JOINT_CHROMHMM] + [utils.method_key(c, joint=True) for c in CALLERS]
# (individual, joint) counterparts of the same binarization.
JOINT_INDIV_PAIRS = list(zip(INTER_DS_METHODS, JOINT_METHODS))
# Every method of a replicate, each individual one before its joint counterpart.
REP_METHODS = INTER_DS_METHODS + JOINT_METHODS
REP_METHOD_LABELS = [utils.display_name(m) for m in REP_METHODS]
REPS = ["rep1", "rep2"]

CHIP_DATASETS = [d for d in DATASETS if not d.endswith("_mint")]
MINT_DATASETS = [d for d in DATASETS if d.endswith("_mint")]
REP_DATASETS = [d for d in DATASETS if DO_REPLICATES and DATASETS[d].get("replicates")]

# Output roots of the cross-dataset plots.
SP = "out/summary_plots"   # cross-dataset summary plots
REF = "out/reference"      # ENCODE reference plots

# (metrics key, display name) of the three metrics and the two modes every
# agreement below is measured in: Full over every state, NOQH with the
# Quies/Het bulk of the genome dropped, since agreeing on it is easy and
# otherwise dominates both kappa and Jaccard.
METRICS = [(KAPPA, KAPPA_DISPLAY), (JACCARD, JACCARD_DISPLAY),
           (COSINE, COSINE_DISPLAY)]
MODES = [(FULL, FULL_DISPLAY), (NOQH, NOQH_DISPLAY)]

# Per-method segmentations shown in per-dataset grids (de-novo + reference).
METHOD_LABELS = [("ref", utils.display_name("ref"))] + [
    (m, utils.display_name(m)) for m in INTER_DS_METHODS]

# Per-dataset detail: the grids and matching matrices of every dataset here are
# rendered by the Display cells of section 1. All 15 of them is a lot of output,
# so the default is the first dataset only; set to list(DATASETS) for all.
DS_DETAIL = list(DATASETS)[:1]

print(f"Datasets      : {len(DATASETS)} "
      f"({len(CHIP_DATASETS)} ChIP-seq, {len(MINT_DATASETS)} Mint-ChIP, "
      f"{len(REP_DATASETS)} with replicates)")
if OUTLIERS:
    print(f"Outliers      : {len(OUTLIERS)} ({', '.join(OUTLIERS)})")
print(f"Callers       : {CALLERS}")
print(f"Match variant : {MATCH_METHOD}")
print(f"Inter methods : {INTER_DS_METHODS}")

In [ ]:
# Path helpers, mirroring the Snakefile functions.
def ds_of(folder):
    """The dataset a folder belongs to: "imr90/rep1" -> "imr90"."""
    return folder.split("/")[0]


def folders_of(ds):
    """The pooled folder of a dataset and, when it has them, its replicates."""
    fl = [ds]
    if ds in REP_DATASETS:
        fl += [f"{ds}/{rep}" for rep in REPS]
    return fl


def ds_title(ds):
    """Plot title of a dataset: its cell type and the assay it was run with."""
    assay = "Mint-ChIP" if ds.endswith("_mint") else "ChIP-seq"
    return f"{DATASETS[ds]['cell']} ({assay})"


def ref_bed_path(ds):
    """The published ENCODE reference segmentation of a dataset."""
    return f"{ds}/{DATASETS[ds]['ref_chromhmm']}_chromhmm.bed"


def dense_bed_path(folder):
    """The default ChromHMM segmentation of a folder, before the matching."""
    cell = DATASETS[ds_of(folder)]["cell"]
    return f"{folder}/{CHROMHMM_DEFAULT}_result/{cell}_{NSTATES}_dense.bed"


def inter_ds_bed(folder, method):
    """Matched segmentation of a de-novo method in a dataset or replicate folder.

    A joint model is not per-folder: one KMeans / ChromHMM model covers both
    replicates of a dataset and writes its own segmentation per replicate under
    {ds}/joint_kmeans and {ds}/joint_chromhmm.
    """
    ds = ds_of(folder)
    sfx = MATCH_METHOD
    if method == CHROMHMM_DEFAULT:
        return dense_bed_path(folder).replace(".bed", f"_{sfx}.bed")
    if method.startswith("joint_"):
        rep = folder.split("/")[-1]
        if method == JOINT_CHROMHMM:
            return f"{ds}/{JOINT_CHROMHMM}/{rep}_{NSTATES}_dense_{sfx}.bed"
        caller = method.replace("joint_kmeans_", "")
        return f"{ds}/joint_kmeans/{caller}/{rep}_kmeans_joint_states_{sfx}.bed"
    caller = method.replace("kmeans_", "")  # omni | homer | macs2
    return f"{folder}/{caller}/{caller}_kmeans_states_{sfx}.bed"


def analysis_dir(folder, method):
    """Where run_analyze() writes the analysis of one matched segmentation."""
    return f"out/{folder}/{MATCH_METHOD}/{method}"


def seg_bin(path):
    """Bin size a segmentation was produced at, from the caller in its path."""
    for caller, size in CALLER_BIN.items():
        if f"/{caller}/" in path:
            return size
    return CHROMHMM_BIN


def existing(paths):
    """Keep only paths that exist on disk (skip segmentations not produced)."""
    return [p for p in paths if os.path.exists(p)]

In [ ]:
# Analysis helpers: what run_analyze() is given per segmentation, and when it
# has to run at all. Shared by the individual segmentations of section 1 and
# the joint ones of section 4.

# The standard ChromHMM annotations every segmentation is enriched against.
annotations = sorted(glob.glob(os.path.join(COORDS_DIR, "*.bed.gz")))
if not annotations:
    print(f"WARNING: No standard annotations found in {COORDS_DIR}!")
    annotations = sorted(glob.glob(os.path.join(TOOLS["coords_dir"], "*.bed.gz")))
    if annotations:
        print(f"Found {len(annotations)} annotations using relative path.")
else:
    print(f"Found {len(annotations)} standard annotations in {COORDS_DIR}")


def ds_annotations(ds, cfg):
    """The standard annotations, plus the dataset's own ATAC-seq peaks."""
    paths = list(annotations)
    if cfg.get("atac"):
        paths.append(f"{ds}/atac_{cfg['atac']}.bed.gz")
    return paths


def rnaseq_args(ds, cfg):
    """run_analyze arguments enabling the RNA-seq annotations, for datasets that have them."""
    if not cfg.get("rnaseq"):
        return {"rnaseq": None, "gtf": None}
    return {"rnaseq": f"{ds}/rnaseq_{cfg['rnaseq']}.tsv", "gtf": GENCODE_GTF}


def needs_enrichment(outdir, cfg):
    """True when run_analyze has to (re)run for the enrichment of *outdir*.

    Either it was never produced, or the dataset has RNA-seq and the enrichment
    predates the expressed/non-expressed gene body annotations.
    """
    path = os.path.join(outdir, "enrichment", "enrichment.tsv")
    if not os.path.exists(path):
        return True
    if not cfg.get("rnaseq"):
        return False
    with open(path) as f:
        return "NonExpressedGeneBodies" not in f.read()


def needs_report(outdir, cfg):
    """True when *outdir* holds no analysis at all; the guard of a segmentation
    analyzed without the functional annotations, which has no enrichment."""
    return not os.path.exists(os.path.join(outdir, "report.tsv"))


def submit_analyze(executor, futures, label, seg, outdir, ds, inputs=None,
                   annotate=True, guard=needs_enrichment):
    """Queue run_analyze() for one segmentation, unless it is up to date.

    *annotate* False leaves out the functional annotations, for a segmentation
    analyzed only for its report and emissions. A segmentation that is not on
    disk is reported and skipped, so a partial pipeline run analyzes what it
    produced.
    """
    cfg = DATASETS[ds]
    if not os.path.exists(seg):
        print(f"  SKIP {label}: no {seg}")
        return
    if not guard(outdir, cfg):
        return
    print(f"Analyzing {label} ...")
    fut = executor.submit(
        analyze.run_analyze,
        seg=seg, bin_size=seg_bin(seg), outdir=outdir, inputs=inputs,
        annotations=ds_annotations(ds, cfg) if annotate else None,
        bw_emissions=seg.replace(".bed", ".bw_emissions.npz"),
        **(rnaseq_args(ds, cfg) if annotate else {"rnaseq": None, "gtf": None}))
    futures[fut] = label


def drain(futures):
    """Wait for every queued analysis, reporting the ones that failed."""
    for fut in as_completed(futures):
        try:
            fut.result()
        except Exception as e:
            print(f"  ERROR {futures[fut]}: {e}")
    print(f"Done ({len(futures)} analyses)")


def comparison_present(outdir):
    """True when run_compare has already written its all-pairs table to *outdir*.

    Existence only: a result is reused whatever produced it, so deleting it is
    how a comparison is recomputed after a change to compare.py.
    """
    return (os.path.exists(os.path.join(outdir, "comparison_all_pairs.tsv")) and
            os.path.exists(os.path.join(outdir, "per_state_metrics.tsv")))


def _try(label, fn):
    """Run one plotting step; report and continue on failure (e.g. missing inputs)."""
    try:
        fn()
    except Exception as e:
        print(f"  SKIP {label}: {e}")

In [ ]:
# Agreement helpers: how two segmentations are compared, shared by the
# replicate agreement of section 4 and the sample-pair sweep of section 5.
def pair_agreement(path1, path2, rematch=False):
    """Agreement of two segmentations per mode, None when they cannot be compared.

    Both modes come out of a single pass over the pair: Full keeps every state,
    NOQH drops the Quies/Het background. A pair that is not on disk, shares no
    state or never overlaps is reported as None and left out of the plots,
    rather than entering them as a row of zeros.

    *rematch* realigns the state space of *path2* onto that of *path1* by
    maximum overlap before the metrics are read - what
    run_compare(rematch=True) does for the inter-dataset comparison. Two
    segmentations of the *same* dataset were matched to the same ENCODE
    reference markup and already share their state names, so they are compared
    as they are; two segmentations of *different* datasets were matched to two
    different markups, where a state name means only as much as the two markups
    agree it does, so they are realigned first.
    """
    if not (os.path.exists(path1) and os.path.exists(path2)):
        return None

    segs1, segs2 = match.load_bed(path1), match.load_bed(path2)
    lengths1, lengths2 = match.state_lengths(segs1), match.state_lengths(segs2)
    if not (set(lengths1) | set(lengths2)):
        return None

    # pair_overlap() puts its work state first, so side 2 goes in as the
    # reference and the keys come out (state1, state2) - the orientation
    # agreement_metrics() reads with lengths1 first.
    overlap = match.pair_overlap(segs2, segs1)
    if sum(overlap.values()) == 0:
        return None

    if rematch:
        # best_mapping() wants the work state first, i.e. (state2, state1).
        mapping = match.best_mapping(
            {(s2, s1): bp for (s1, s2), bp in overlap.items()},
            sorted(lengths2, key=analyze._natural_sort_key),
            sorted(lengths1, key=analyze._natural_sort_key))
        remapped = defaultdict(float)
        for (s1, s2), bp in overlap.items():
            remapped[(s1, mapping[s2])] += bp
        overlap = remapped
        lengths2 = {mapping[s2]: bp for s2, bp in lengths2.items()}

    return match.agreement_by_mode(overlap, lengths1, lengths2,
                                   background=NOQH_STATES)


def agreement_row(method, unit, modes, **extra):
    """The rows one compared pair contributes: one per mode, plus *extra* flags."""
    print(f"  {unit} {method}: " + ", ".join(
        f"{name} " + " ".join(f"{m}={modes[key][m]:.3f}" for m, _ in METRICS)
        for key, name in MODES))
    return [{"Method": utils.display_name(method), "Dataset": unit, "Mode": name,
             **extra,
             **{utils.metric_display(m): v for m, v in modes[key].items()}}
            for key, name in MODES]


def agreement_table(pairs, rematch=False):
    """DataFrame of pair_agreement() over (method, unit, path1, path2) tuples.

    One row per pair and mode, so a plot selects its mode by filtering on Mode -
    the shape the caches summary.ipynb reads are written in.
    """
    rows = []
    for method, unit, path1, path2 in pairs:
        modes = pair_agreement(path1, path2, rematch=rematch)
        if modes is None:
            print(f"  SKIP {unit} {method}: {path1} vs {path2}")
            continue
        rows += agreement_row(method, unit, modes)
    return pd.DataFrame(rows)


def has_modes(df, columns=()):
    """Cache guard: an agreement table holds both modes of every pair it kept.

    Also rejects a cache from before the modes, the metrics or one of
    *columns* were added, and an empty one, which means the segmentations were
    not on disk yet.
    """
    return (not df.empty
            and set(df.get("Mode", [])) == {name for _, name in MODES}
            and all(c in df.columns for c in columns))


def mean_agreement(df, label, mask=None):
    """Print the mean NOQH kappa per method over *df*, as a run-time summary."""
    noqh = df[df["Mode"] == NOQH_DISPLAY]
    if mask is not None:
        noqh = noqh[mask.reindex(noqh.index, fill_value=False)]
    if noqh.empty:
        return
    means = noqh.groupby("Method")[KAPPA_DISPLAY].mean().round(3)
    counts = noqh.groupby("Method")["Dataset"].nunique().to_dict()
    print(f"  {NOQH_DISPLAY} {KAPPA} over {label}: {means.to_dict()}")
    print(f"    pairs per method: {counts}")

In [ ]:
# Plotting helpers: the method bar plots sections 4-6 draw. One bar per method,
# mean +- SE across the units behind it, every observation shown, and the joint
# models hatched, as everywhere else in the project.
def methods_present(df, methods):
    """(method key, display name) of every method with data, in the given order."""
    present = set(df["Method"])
    return [(m, utils.display_name(m)) for m in methods
            if utils.display_name(m) in present]


def annotate_means(ax, df, x, order, metric, fmt):
    """Print the mean of every bar above it."""
    yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
    for i, name in enumerate(order):
        vals = df[df[x] == name][metric]
        if vals.empty:
            continue
        ax.text(i, vals.max() + yrange * 0.01, fmt.format(vals.mean()),
                ha="center", va="bottom", fontsize=6)


def method_barplot(df, metric, methods, title, outfile,
                   ylabel=None, ylim=(0, 1.05), fmt="{:.2f}",
                   point_size=utils.STRIP_SIZE, point_alpha=None,
                   n_per_method=False):
    """Bar plot of one metric per method; methods without data are dropped.

    point_size / point_alpha thin the observation overlay down for a bar that
    carries many of them - a cross-sample bar holds every dataset pair.

    n_per_method puts each bar's own observation count on its tick label
    instead of one count in the title, for a plot whose methods do not all
    cover the same units: a joint ENCODE model reaches only the dataset pairs
    that have replicates, so a single n would be wrong for half the bars.
    """
    if df.empty:
        print(f"  SKIP {outfile}: no data")
        return
    shown = methods_present(df, methods)
    if not shown:
        print(f"  SKIP {outfile}: none of {methods} present")
        return
    order = [label for _, label in shown]

    fig, ax = plt.subplots(figsize=(max(7, len(order) * 1.3), 5))
    sns.barplot(data=df, x="Method", y=metric, order=order,
                hue="Method", hue_order=order, dodge=False, legend=False,
                palette={label: utils.method_color(m) for m, label in shown},
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1, ax=ax)
    utils.hatch_joint(ax, order)
    utils.strip_points(ax, data=df, x="Method", y=metric, order=order,
                       size=point_size,
                       **({} if point_alpha is None else {"alpha": point_alpha}))
    if n_per_method:
        counts = df.groupby("Method")["Dataset"].nunique()
        ax.set_title(title, fontsize=11, fontweight="bold")
    else:
        ax.set_title(f"{title} (n={df['Dataset'].nunique()})", fontsize=11,
                     fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel or metric, fontsize=9)
    if ylim:
        # A NOQH kappa can go negative; keep such a bar inside the frame.
        ax.set_ylim(min(ylim[0], float(df[metric].min())), ylim[1])
    ax.grid(axis="y", alpha=0.3)
    if n_per_method:
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels([f"{label} (n={counts.get(label, 0)})"
                            for label in order])
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    annotate_means(ax, df, "Method", order, metric, fmt)
    utils.save_fig(fig, outfile)


def state_barplot(df, metric, methods, title, outfile, ylabel=None):
    """Per-state bar plot of one metric, one bar per method within each state."""
    if df.empty:
        print(f"  SKIP {outfile}: no data")
        return
    shown = methods_present(df, methods)
    if not shown:
        print(f"  SKIP {outfile}: none of {methods} present")
        return
    order = [label for _, label in shown]
    states = summary_plots.sort_states(df["State"].unique())

    fig, ax = plt.subplots(figsize=(max(10, len(states) * 0.9), 5))
    sns.barplot(data=df, x="State", y=metric, order=states,
                hue="Method", hue_order=order,
                palette={label: utils.method_color(m) for m, label in shown},
                capsize=0.05, errorbar="se", err_kws={"linewidth": 1.0},
                edgecolor="lightgrey", linewidth=0.5, ax=ax)
    utils.hatch_joint(ax, order)
    utils.strip_points(ax, data=df, x="State", y=metric,
                       hue="Method", order=states, hue_order=order)
    ax.set_title(f"{title} (n={df['Dataset'].nunique()})", fontsize=11,
                 fontweight="bold")
    ax.set_xlabel("Chromatin state", fontsize=9)
    ax.set_ylabel(ylabel or metric, fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.legend(title="Method", fontsize=8, title_fontsize=9,
              bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
    utils.save_fig(fig, outfile)


def mode_barplots(df, methods, title, outdir, prefix, **kwargs):
    """The six bar plots of an agreement table: three metrics x two modes."""
    for metric, label in METRICS:
        for key, mode in MODES:
            suffix = "" if key == FULL else f"_{key}"
            method_barplot(df[df["Mode"] == mode], label, methods,
                           f"{title} ({label}, {mode})",
                           f"{outdir}/{prefix}_{metric}{suffix}.png",
                           ylabel=f"{label} ({mode})", **kwargs)


def mode_items(outdir, prefix, caption):
    """(path, caption) of the six plots mode_barplots() wrote, for show_group()."""
    return [(f"{outdir}/{prefix}_{metric}{'' if key == FULL else f'_{key}'}.png",
             f"{label}, {caption} - {mode}")
            for metric, label in METRICS for key, mode in MODES]

# 1. Segmentation analysis and cross-dataset summaries

The per-segmentation analysis of every markup the pipeline produced, the comparisons
between them, and the cross-dataset summary plots read off the tables both write.

### 1.1 Compute — per-segmentation analysis

`analyze_peaks.run_analyze_peaks` for the binarization statistics of a dataset, and
`analyze.run_analyze` per segmentation for its report, transition entropy, emissions
and functional enrichment. Every segmentation is analyzed once: a folder whose
analysis is already on disk is skipped, so a re-run only fills gaps.

In [ ]:
# 1.1 Compute: peak statistics per dataset, then every individual segmentation
# of every dataset and replicate folder - the ENCODE reference, the default
# ChromHMM segmentation before and after the matching, and the KMeans
# segmentation of each peak caller. The joint models are analyzed the same way
# in section 4, off the same submit_analyze() helper.
futures = {}
with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    for ds, cfg in DATASETS.items():
        peaks_outdir = f"{ds}/peaks"
        if not os.path.exists(os.path.join(peaks_outdir, "peak_stats.tsv")):
            print(f"Analyzing peaks for {ds} ...")
            futures[executor.submit(
                analyze_peaks.run_analyze_peaks,
                ds=ds, cell=cfg["cell"], marks=list(MARKS), outdir=peaks_outdir,
                omni_bin=CALLER_BIN[OMNI], chromhmm_bin=CHROMHMM_BIN)] = f"peaks {ds}"

        for folder in folders_of(ds):
            submit_analyze(executor, futures, f"reference in {folder}",
                           ref_bed_path(ds), f"out/{folder}/ref", ds)

            # The default ChromHMM segmentation before the matching: analyzed
            # for its report and emissions only, so no annotations.
            submit_analyze(executor, futures, f"default ChromHMM in {folder}",
                           dense_bed_path(folder),
                           f"out/{folder}/chromhmm_default_dense", ds,
                           inputs=[f"{folder}/{CHROMHMM_DEFAULT}/*.txt"],
                           annotate=False, guard=needs_report)

            submit_analyze(executor, futures, f"matched ChromHMM in {folder}",
                           inter_ds_bed(folder, CHROMHMM_DEFAULT),
                           analysis_dir(folder, CHROMHMM_DEFAULT), ds,
                           inputs=[f"{folder}/{CHROMHMM_DEFAULT}/*.txt"])

            for caller in CALLERS:
                method = utils.method_key(caller)
                submit_analyze(executor, futures, f"KMeans {caller} in {folder}",
                               inter_ds_bed(folder, method),
                               analysis_dir(folder, method), ds,
                               inputs=[f"{folder}/{caller}/chromhmm_peaks/chr*.txt.gz"])

drain(futures)

### 1.2 Compute — per-dataset comparison

Per dataset: transition-matrix entropy, pairwise Cohen's κ and Jaccard similarity,
emission similarity and segment-length statistics of its own segmentations, plus the
unified method comparison table. Reads the per-segmentation analysis of 1.1.

In [ ]:
# 1.2 Compute: every segmentation of a dataset against the others, then the
# aggregated per-method table and plots of that dataset.
def ds_compare_segs(ds):
    """The ENCODE reference and every de-novo segmentation of one dataset."""
    return [ref_bed_path(ds)] + [inter_ds_bed(folder, method)
                                 for folder in folders_of(ds)
                                 for method in INTER_DS_METHODS]


dataset_args = []
for ds in DATASETS:
    segs = existing(ds_compare_segs(ds))
    if len(segs) < 2:
        print(f"{ds}: found {len(segs)} segmentation(s), skipping comparison")
        continue
    dataset_args.append((ds, segs, [seg_bin(p) for p in segs]))

print(f"Comparing {len(dataset_args)} datasets (variant={MATCH_METHOD}) ...")

# 1. run_compare: cross-segmentation similarity metrics and statistics.
# Note: run_compare internally parallelizes pair comparisons.
for ds, segs, bins in dataset_args:
    outdir = f"out/{ds}/{MATCH_METHOD}"
    if comparison_present(outdir):
        print(f"  {ds} comparison already present, skipping.")
        continue
    print(f"  {ds} ...")
    _try(f"compare {ds}", lambda segs=segs, bins=bins, outdir=outdir:
         compare.run_compare(seg=segs, bins=bins, outdir=outdir,
                             analysis_dir=outdir, rematch=True))

# 2. run_compare_methods: aggregate the results into the per-method table and plots.
for ds, segs, bins in dataset_args:
    outdir = f"out/{ds}/{MATCH_METHOD}"
    table_path = os.path.join(outdir, "comparison_table.tsv")
    pairs_path = os.path.join(outdir, "comparison_all_pairs.tsv")
    if os.path.exists(table_path):
        # A table from before the ATAC-seq metrics has to be rebuilt, and so
        # has one older than the comparison it aggregates: deleting a result of
        # the loop above to recompute it leaves this table on the numbers that
        # result used to hold.
        try:
            cols = pd.read_csv(table_path, sep="\t", nrows=0).columns
            current = (not os.path.exists(pairs_path)
                       or os.path.getmtime(table_path) >= os.path.getmtime(pairs_path))
            if (current and "enrich_Active_ATAC" in cols
                    and "enrich_Active_NonExpGeneBodies" in cols):
                print(f"  {ds} compare methods already present, skipping.")
                continue
        except Exception:
            pass
    print(f"  {ds} compare methods ...")
    _try(f"compare_methods {ds}", lambda outdir=outdir, ds=ds:
         compare_methods.run_compare_methods(
             analysis_dir=outdir, comparison_dir=outdir, outdir=outdir,
             ref_dir=f"out/{ds}"))

### 1.3 Compute — inter-dataset and reference comparison

The same method across datasets (`out/{method}`), aggregated into one cross-dataset
table, and the published ENCODE reference segmentations against each other
(`out/reference`). These are the name-matched, bin-level comparisons the similarity
matrices and distributions are read off; section 5 scores the same sample pairs on
realigned raw states and covers the joint models as well.

In [ ]:
# 1.3 Compute: inter-dataset comparison per method, and the ENCODE references
# against each other.
ds_list = list(DATASETS)
cells = [DATASETS[d]["cell"] for d in ds_list]
os.makedirs(SP, exist_ok=True)

# 1. Per-method cross-dataset comparison (every dataset pair).
for method in INTER_DS_METHODS:
    pairs = [(d, inter_ds_bed(d, method)) for d in ds_list]
    pairs = [(d, p) for d, p in pairs if os.path.exists(p)]
    if len(pairs) < 2:
        print(f"  SKIP inter compare {method}: <2 datasets with this segmentation")
        continue

    outdir = f"out/{method}"
    if comparison_present(outdir):
        print(f"  {method} inter-dataset comparison already present, skipping.")
        continue

    segs = [p for _, p in pairs]
    labels = [f"{d}:{method}" for d, _ in pairs]
    print(f"Inter-dataset compare: {method} ({len(segs)} datasets)")
    _try(f"compare {method}",
         lambda segs=segs, labels=labels, outdir=outdir:
         compare.run_compare(seg=segs, bins=[seg_bin(p) for p in segs],
                             labels=labels, all_pairs=True,
                             outdir=outdir, rematch=True))

# 2. Aggregate the per-method kappa matrices into one cross-dataset table,
# which summary.ipynb reads as the cross-sample evidence of the individual models.
if not os.path.exists("out/comparison_table.tsv"):
    _try("comparison_table", lambda: compare_out.run_compare_out(
        methods=INTER_DS_METHODS, indir="out",
        outfile="out/comparison_table.tsv"))

# 3. Pairwise similarity among all ENCODE reference segmentations.
ref_segs = [(ds, ref_bed_path(ds)) for ds in ds_list if DATASETS[ds].get("ref_chromhmm")]
ref_segs = [(ds, p) for ds, p in ref_segs if os.path.exists(p)]
ref_paths = [p for _, p in ref_segs]
ref_labels = [DATASETS[ds]["cell"] for ds, _ in ref_segs]

if not ref_paths:
    print("No reference markups found for datasets in config_encode.yaml")
elif comparison_present(REF):
    print("Reference comparison already present, skipping.")
else:
    print(f"Reference compare ({len(ref_paths)} segmentations) ...")
    _try("reference compare", lambda: compare.run_compare(
        seg=ref_paths, bins=CHROMHMM_BIN, labels=ref_labels, all_pairs=True,
        outdir=REF, rematch=True))

### 1.4 Plotting — cross-dataset summaries

`summary_plots` over the tables 1.1-1.3 wrote: the reference plots under
`out/reference`, the cross-dataset bar and distribution plots under
`out/summary_plots`, the per-assay (ChIP-seq / Mint-ChIP) splits of them under
`chip/` and `mint/`, and the emission discriminability summary.

In [ ]:
# 1.4 Plotting: the ENCODE reference summaries.
_try("reference summary plots", lambda: summary_plots.run_summary_plots(
    ref_paths=ref_paths, ref_labels=ref_labels,
    ref_composition_outfile=f"{REF}/state_composition.png",
    ref_comp_matrix=f"{REF}/composition_similarity_matrix.tsv",
    ref_kappa_matrix=f"{REF}/kappa_matrix.tsv",
    ref_jaccard_matrix=f"{REF}/jaccard_similarity_matrix.tsv",
    ref_dist_outfile=f"{REF}/similarity_distribution.png",
    ref_comp_noqh_matrix=f"{REF}/composition_noqh_similarity_matrix.tsv",
    ref_kappa_noqh_matrix=f"{REF}/kappa_noqh_matrix.tsv",
    ref_jaccard_noqh_matrix=f"{REF}/jaccard_noqh_matrix.tsv",
    ref_dist_noqh_outfile=f"{REF}/similarity_distribution_noqh.png"))

In [ ]:
# 1.4 Plotting: the cross-dataset summary bar and distribution plots. Every
# dataset's analysis and comparison live in the same directory, so one list of
# dirs serves as both the analysis and the methods source.
ds_dirs = [f"out/{d}/{MATCH_METHOD}" for d in ds_list]

_try("summary bar plots", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, methods_dirs=ds_dirs, analysis_dirs=ds_dirs,
    methods=INTER_DS_METHODS, outdir=SP))

_try("state coverage", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, ref_paths=ref_paths,
    methods=INTER_DS_METHODS, nstates=NSTATES, match_method=MATCH_METHOD,
    state_coverage_outfile=f"{SP}/state_coverage.png"))

_try("peak stats", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, workdir=workdir, methods=INTER_DS_METHODS,
    peak_count_outfile=f"{SP}/n_peaks.png",
    peak_length_outfile=f"{SP}/peak_length.png"))

_try("method similarity distribution", lambda: summary_plots.run_summary_plots(
    method_sim_dist_indir="out", method_sim_dist_methods=INTER_DS_METHODS,
    method_sim_dist_outfile=f"{SP}/method_similarity_distribution.png",
    method_sim_dist_noqh_outfile=f"{SP}/method_similarity_distribution_noqh.png"))

if CHIP_DATASETS and MINT_DATASETS:
    _try("ChIP vs Mint similarity distribution", lambda: summary_plots.run_summary_plots(
        method_sim_dist_indir="out", method_sim_dist_methods=INTER_DS_METHODS,
        method_sim_dist_group_a=CHIP_DATASETS, method_sim_dist_group_b=MINT_DATASETS,
        method_sim_dist_filtered_outfile=f"{SP}/method_similarity_distribution_chip_vs_mint.png",
        method_sim_dist_filtered_noqh_outfile=f"{SP}/method_similarity_distribution_chip_vs_mint_noqh.png"))

if REP_DATASETS:
    _try("replicate consistency", lambda: summary_plots.run_summary_plots(
        datasets=REP_DATASETS,
        methods_dirs=[f"out/{d}/{MATCH_METHOD}" for d in REP_DATASETS],
        methods=INTER_DS_METHODS, rep_consistency_outdir=SP))

_try("per-dataset state composition", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, methods=INTER_DS_METHODS,
    nstates=NSTATES, match_method=MATCH_METHOD,
    method_ds_composition_outdir=SP,
    all_methods_composition_outdir=SP))

_try("mean state composition", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, ref_paths=ref_paths,
    methods=INTER_DS_METHODS, nstates=NSTATES, match_method=MATCH_METHOD,
    method_composition_outfile=f"{SP}/method_state_composition.png"))

# Per-assay (ChIP-seq vs Mint-ChIP) splits of the cross-dataset summaries:
# the peaks, total-segments and per-dataset ENCODE-reference plots again,
# separately per assay group, into the chip/ and mint/ subdirs of SP.
for grp, gname, dss in [("chip", "ChIP-seq", CHIP_DATASETS),
                        ("mint", "Mint-ChIP", MINT_DATASETS)]:
    if not dss:
        continue
    gdir = f"{SP}/{grp}"
    os.makedirs(gdir, exist_ok=True)
    gdirs = [f"out/{d}/{MATCH_METHOD}" for d in dss]
    _try(f"summary bars [{grp}]", lambda dss=dss, gdirs=gdirs, gdir=gdir:
         summary_plots.run_summary_plots(
             datasets=dss, methods_dirs=gdirs, analysis_dirs=gdirs,
             methods=INTER_DS_METHODS, outdir=gdir,
             all_methods_composition_outdir=gdir))
    _try(f"peaks [{grp}]", lambda dss=dss, gdir=gdir:
         summary_plots.run_summary_plots(
             datasets=dss, workdir=workdir, methods=INTER_DS_METHODS,
             peak_count_outfile=f"{gdir}/n_peaks.png",
             peak_length_outfile=f"{gdir}/peak_length.png"))
    _try(f"reference n_segments [{grp}]", lambda dss=dss, gdirs=gdirs, gdir=gdir, gname=gname:
         summary_plots.plot_reference_n_segments(
             dss, gdirs, [DATASETS[d]["cell"] for d in dss],
             f"{gdir}/reference_n_segments.png",
             f"ENCODE reference segments — {gname}"))

In [ ]:
# 1.4 Plotting: emission discriminability (Gini) per dataset and summary.
if not os.path.exists(f"{SP}/emission_gini_summary.png"):
    _try("emission similarity", lambda: emission_similarity.run_emission_similarity(
        datasets=ds_list, analysis_dirs=[f"out/{d}/{MATCH_METHOD}" for d in ds_list],
        methods=INTER_DS_METHODS, outdir=SP))

### 1.5 Display — results

Every figure the cells above produced, grouped into:

1. **Peaks** — number and lengths
2. **Segmentation** — number of states and segments
3. **Segmentation** — state lengths
4. **Segmentation** — state composition
5. **Other analyses** — transition entropy, agreement against the ENCODE reference,
   biological validation, emission discriminability and the comparison table
6. **Per-dataset detail** — the analysis of every segmentation of the datasets
   `DS_DETAIL` names, and the work-state → reference matching matrices

Replicate consistency is shown in section 4 and the cross-sample and cross-assay
agreement in section 5, next to the rest of their material.

#### 1.5.1 Peaks — number and lengths

Binarization peak statistics: peak count and mean peak length per mark and method,
summarised across datasets and per assay group.

In [ ]:
# 1.5.1 Peaks — number and lengths
PEAK_ITEMS = [("n_peaks.png", "Peak count per mark/method — mean ± std across datasets"),
              ("peak_length.png", "Mean peak length per mark/method — mean ± std across datasets")]

show_group("Cross-dataset summary",
           [(f"{SP}/{name}", cap) for name, cap in PEAK_ITEMS], level=2)

for grp, gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"Cross-dataset summary — {gname}",
               [(f"{SP}/{grp}/{name}", cap) for name, cap in PEAK_ITEMS], level=2)

#### 1.5.2 Segmentation — number of states and segments

How fragmented each segmentation is: the total segment count per method across
datasets, and the per-dataset ENCODE reference state and segment counts.

In [ ]:
# 1.5.2 Segmentation — number of states and segments
for grp, gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"Cross-dataset summary — {gname}", [
        (f"{SP}/{grp}/summary_n_segments.png",
         "Total number of segments per method (incl. ENCODE reference) — mean ± std across datasets"),
    ], level=2)

for grp, gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"ENCODE reference segmentations — {gname}", [
        (f"{SP}/{grp}/reference_n_segments.png",
         "Number of segments per dataset's ENCODE reference"),
    ])

#### 1.5.3 Segmentation — state lengths

Segment length distributions: the cross-dataset per-state coverage and the ENCODE
reference length summaries.

In [ ]:
# 1.5.3 Segmentation — state lengths
show_group("Cross-dataset summary", [
    (f"{SP}/state_coverage.png", "Genomic coverage fraction per chromatin state"),
    (f"{SP}/summary_mean_tx_length.png", "Mean Tx (transcription) segment length"),
], level=2)

show_group("ENCODE reference segmentations", [
    (f"{REF}/mean_length.png", "Mean segment length"),
    (f"{REF}/median_length.png", "Median segment length"),
    (f"{REF}/min_length.png", "Min segment length"),
    (f"{REF}/max_length.png", "Max segment length"),
], width=750)

#### 1.5.4 Segmentation — state composition

Fraction of the genome covered by each chromatin state: averaged per method across
datasets, across the ENCODE reference segmentations, and per dataset for each
de-novo method.

In [ ]:
# 1.5.4 Segmentation — state composition
show_group("Per-method composition (mean across datasets)", [
    (f"{SP}/method_state_composition.png", "State composition per method — mean across datasets"),
], level=2)

show_group("ENCODE reference composition", [
    (f"{REF}/state_composition.png", "State composition across ENCODE reference segmentations"),
])

show_group("Per-dataset composition for each de-novo method",
           [(p, os.path.basename(p).replace("method_ds_composition_", "").replace(".png", ""))
            for p in sorted(glob.glob(f"{SP}/method_ds_composition_*.png"))])

show_group("Per-method composition for each dataset",
           [(p, os.path.basename(p).replace("ds_composition_", "").replace(".png", ""))
            for p in sorted(glob.glob(f"{SP}/ds_composition_*.png"))])

#### 1.5.5 Other analyses

Transition entropy, agreement against the ENCODE reference, biological validation
against RNA-seq and ATAC-seq, emission discriminability and the aggregated
cross-dataset comparison table.

In [ ]:
# 1.5.5 Other analyses
show_group("Transition matrix entropy", [
    (f"{SP}/summary_entropy.png", "De-novo — Full (raw labels)"),
    (f"{SP}/summary_entropy_noqh.png", "De-novo — NOQH (excl. Quies/Het)"),
    (f"{REF}/entropy_summary_combined.png", "ENCODE reference entropy (Full + NOQH)"),
], level=2)

show_group("Agreement vs ENCODE reference", [
    (f"{SP}/summary_kappa_vs_ref.png", "Kappa agreement vs ENCODE reference"),
    (f"{SP}/summary_jaccard_vs_ref.png", "Jaccard agreement vs ENCODE reference"),
    (f"{SP}/summary_cosine_vs_ref.png", "Cosine agreement vs ENCODE reference"),
    (f"{SP}/per_state_kappa_bar.png", "Per-state Kappa vs ENCODE reference — combined across methods"),
    (f"{SP}/per_state_jaccard_bar.png", "Per-state Jaccard vs ENCODE reference — combined across methods"),
    (f"{SP}/per_state_kappa_summary.png", "Per-state Kappa vs ENCODE reference (Summary)"),
    (f"{SP}/per_state_jaccard_summary.png", "Per-state Jaccard vs ENCODE reference (Summary)"),
], level=2)

show_group("Biological validation (RNA-seq / ATAC-seq)", [
    (f"{SP}/summary_jaccard_tx.png", "Jaccard: Tx state vs expressed gene bodies"),
    (f"{SP}/summary_enrich_tx.png", "Tx fold enrichment at expressed gene bodies"),
    (f"{SP}/summary_sensitivity_tx.png", "Fraction of expressed gene bodies covered by Tx states"),
    (f"{SP}/summary_coverage_tx.png", "Fraction of Tx states covered by expressed genes"),
    (f"{SP}/summary_2way_tx.png", "2-way validation: Tx state vs expressed gene bodies"),
    (f"{SP}/summary_jaccard_tss.png", "Jaccard: Tss state vs RefSeq TSS ±2 kb"),
    (f"{SP}/summary_enrich_tss.png", "Tss fold enrichment at RefSeq TSS ±2 kb"),
    (f"{SP}/summary_sensitivity_tss.png", "Fraction of RefSeq TSS ±2 kb covered by Tss states"),
    (f"{SP}/summary_coverage_tss.png", "Fraction of Tss states covered by RefSeq TSS ±2 kb"),
    (f"{SP}/summary_2way_tss.png", "2-way validation: Tss state vs RefSeq TSS ±2 kb"),
    (f"{SP}/summary_jaccard_tss_exptss.png", "Jaccard: Tss state vs Expressed TSS"),
    (f"{SP}/summary_jaccard_tss_exptss2kb.png", "Jaccard: Tss state vs Expressed TSS ±2 kb"),
    (f"{SP}/summary_enrich_tss_exptss2kb.png", "Tss fold enrichment at Expressed TSS ±2 kb"),
    (f"{SP}/summary_sensitivity_tss_exptss.png", "Fraction of Expressed TSS covered by Tss states"),
    (f"{SP}/summary_coverage_tss_exptss.png", "Fraction of Tss states covered by Expressed TSS"),
    (f"{SP}/summary_2way_tss_exptss.png", "2-way validation: Tss state vs Expressed TSS"),
    (f"{SP}/summary_2way_tss_exptss2kb.png", "2-way validation: Tss state vs Expressed TSS ±2 kb"),
    (f"{SP}/summary_jaccard_active_atac.png", "Jaccard: Active states vs ATAC-seq"),
    (f"{SP}/summary_enrich_active_atac.png", "Active chromatin enrichment at ATAC-seq peaks"),
    (f"{SP}/summary_sensitivity_active_atac.png", "Fraction of ATAC-seq peaks covered by Active states"),
    (f"{SP}/summary_coverage_active_atac.png", "Fraction of Active states covered by ATAC-seq peaks"),
    (f"{SP}/summary_2way_active_atac.png", "2-way validation: Active chromatin vs ATAC-seq"),
    (f"{SP}/summary_sensitivity_tss_atac.png", "Fraction of ATAC-seq peaks covered by Tss states"),
    (f"{SP}/summary_sensitivity_enh_atac.png", "Fraction of ATAC-seq peaks covered by Enh states"),
    (f"{SP}/summary_enrich_quies_atac.png", "Quiescent states enrichment at ATAC-seq peaks (depletion)"),
    (f"{SP}/summary_enrich_active_nonexp.png", "Active states enrichment at Non-expressed Gene Bodies (depletion)"),
    (f"{SP}/summary_enrich_quies_nonexp.png", "Quiescent states enrichment at Non-expressed Gene Bodies"),
], level=2)

show_group("Emission discriminability (Gini index)", [
    (f"{SP}/emission_gini_summary.png", "Gini index of state emissions"),
], level=2)

header("Cross-dataset comparison table", 2)
if not show_table("out/comparison_table.tsv"):
    print("  (no aggregated comparison table)")

#### 1.5.6 Per-dataset detail

The analysis of every segmentation of the datasets `DS_DETAIL` names: peak statistics,
segment counts and lengths, the method comparison table, the functional enrichment and
emission plots, and the work-state → ENCODE-reference matching score matrices produced
by `match.py` (rows are the de-novo method's states, columns the reference states, the
Hungarian-selected match per row outlined in red).

`DS_DETAIL` is set in the Setup cell and holds the first dataset by default; set it to
`list(DATASETS)` for all of them.

In [ ]:
# 1.5.6 Per-dataset detail
for ds in DS_DETAIL:
    show_group(ds_title(ds), [
        (f"{ds}/peaks/n_peaks.png", "Number of peaks per mark"),
        (f"{ds}/peaks/mean_length.png", "Mean peak length per mark"),
        (f"{ds}/peaks/median_length.png", "Median peak length per mark"),
        (f"{ds}/peaks/jaccard_rep1_vs_rep2.png", "Peak Jaccard: rep1 vs rep2"),
    ], width=600, level=3)

    show_group(ds_title(ds), [
        (f"out/{ds}/{MATCH_METHOD}/n_segments.png", "Number of segments per segmentation"),
        (f"out/{ds}/{MATCH_METHOD}/mean_Tx_length.png", "Mean Tx length per segmentation"),
        (f"{SP}/per_state_kappa_{ds}.png", "Per-state Cohen's Kappa vs ENCODE reference"),
    ], width=750, level=3)

    show_group(ds_title(ds), [
        (f"out/{ds}/{MATCH_METHOD}/{name}_length.png", f"{name.capitalize()} segment length per segmentation")
        for name in ("mean", "median", "min", "max")
    ], width=750, level=3)

    show_group(f"{ds_title(ds)} — per-method length distribution",
               [(method_plot(ds, k, "segment_length.png", MATCH_METHOD), lbl)
                for k, lbl in METHOD_LABELS], width=600, level=4)

    header(ds_title(ds), 3)
    show_table(f"out/{ds}/{MATCH_METHOD}/comparison_table.tsv",
               caption="Method comparison table")
    for k, lbl in METHOD_LABELS:
        show_group(lbl, [
            (method_plot(ds, k, "enrichment/enrichment.png", MATCH_METHOD),
             f"{lbl}: functional enrichment"),
            (method_plot(ds, k, "bin_emissions/state_emissions.png", MATCH_METHOD),
             f"{lbl}: binarized emissions"),
            (method_plot(ds, k, "bw_emissions/state_emissions.png", MATCH_METHOD),
             f"{lbl}: bigwig emissions"),
        ], width=760, level=4)

    show_group(f"{ds_title(ds)} — per-state matching matrices (work → ENCODE reference)",
               [(inter_ds_bed(ds, m).replace(".bed", ".match.png"),
                 f"{utils.display_name(m)}: per-state matching score "
                 f"(matched cell outlined in red)")
                for m in INTER_DS_METHODS], width=620, level=3)

# 2. Optimal number of states

How many states the data supports, per binarization:

1. **Elbow method (inertia)** — the point where the decrease in inertia slows down
2. **Silhouette score** — higher means better-defined clusters
3. **Transition matrix entropy** — lower means more predictable transitions
4. **Realized states** — how many of the requested states a model actually uses

Every sweep runs on chr22 only and is cached per dataset and method under
`out/n_states`, so a re-run fills gaps rather than recomputing.

### 2.1 Compute

In [ ]:
# 2.1 Compute: KMeans over the chr22 binarization of each peak caller, and
# ChromHMM LearnModel on chr22, for every requested state count.
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import subprocess

n_states_range = range(2, 71)
sample_size = 10_000
n_states_results_path = "out/optimal_n_states.tsv"
os.makedirs("out/n_states", exist_ok=True)
SWEEP_COLUMNS = ["inertia", "silhouette", "entropy", "actual_n_states"]

# (sweep key, method key) of every swept binarization: the ChromHMM one, whose
# sweep re-learns the model, and one KMeans sweep per peak caller.
SWEEP_METHODS = [(CHROMHMM, CHROMHMM_DEFAULT)] + [(c, utils.method_key(c)) for c in CALLERS]
SWEEP_METHOD_KEY = dict(SWEEP_METHODS)


class MissingInput(Exception):
    """Raised by a sweep whose binarized chr22 input is not on disk."""


def sweep_label(sweep):
    """Filename part of a sweep: the caller, under its display name."""
    return OMNI_DISPLAY.lower() if sweep == OMNI else sweep


def sweep_cache_path(ds, sweep):
    return f"out/n_states/{ds}_{sweep}.tsv"


def sweep_df(**columns):
    """A sweep as one row per n_states; metrics a method does not have are NaN."""
    blank = [np.nan] * len(n_states_range)
    return pd.DataFrame({"n_states": list(n_states_range),
                         **{c: columns.get(c, blank) for c in SWEEP_COLUMNS}})


def sweep_valid(df):
    """A cached sweep is reusable once every n_states has an entropy."""
    return ("n_states" in df.columns and "entropy" in df.columns
            and set(df["n_states"]) == set(n_states_range)
            and not df["entropy"].isna().any())


def entropy_of_labels(labels, bin_size):
    """Transition entropy of a chr22 label vector, as a segmentation would give."""
    segs = [["chr22", i * bin_size, (i + 1) * bin_size, str(label)]
            for i, label in enumerate(labels)]
    states, counts, state_bp = analyze.build_transition_matrix(segs, bin_size)
    if not states:
        return 0
    total_H, _, _, _ = analyze.transition_entropy(states, counts, state_bp)
    return total_H


def kmeans_sweep(ds, caller):
    """KMeans on the chr22 binarization of *caller*, for every n in n_states_range."""
    cell = DATASETS[ds]["cell"]
    peaks_dir = os.path.join(ds, caller, "chromhmm_peaks")
    data_path = next((p for p in [os.path.join(peaks_dir, f"{name}_chr22_binary.txt.gz")
                                  for name in (cell, ds)]
                      + [os.path.join(peaks_dir, "chr22_binary.txt.gz")]
                      if os.path.exists(p)), None)
    if not data_path:
        raise MissingInput(f"no chr22 binarization for {ds} {caller}")

    print(f"Loading data for {ds} {caller} from {data_path} ...")
    chrom, marks, X = analyze.load_binary(data_path)
    bin_size = CALLER_BIN[caller]

    if X.shape[0] > sample_size:
        np.random.seed(42)
        idx = np.random.choice(X.shape[0], sample_size, replace=False)
        X_sample = X[idx]
    else:
        X_sample = X

    inertia, silhouette_scores, entropies, actual_n_states = [], [], [], []
    base_inertia = max(1, np.sum((X - X.mean(axis=0)) ** 2))

    for n in n_states_range:
        kmeans = KMeans(n_clusters=n, init='k-means++', random_state=42, n_init=10)
        kmeans.fit(X)
        inertia.append(kmeans.inertia_ / base_inertia)
        silhouette_scores.append(silhouette_score(X_sample, kmeans.predict(X_sample)))
        entropies.append(entropy_of_labels(kmeans.labels_, bin_size))
        actual_n_states.append(len(np.unique(kmeans.labels_)))

    return sweep_df(inertia=inertia, silhouette=silhouette_scores,
                    entropy=entropies, actual_n_states=actual_n_states)


def chromhmm_sweep(ds):
    """ChromHMM LearnModel on chr22, for every n in n_states_range."""
    cell = DATASETS[ds]["cell"]
    chr22_indir = os.path.join(ds, "chromhmm_chr22")
    os.makedirs(chr22_indir, exist_ok=True)
    src = None
    for indir in ([os.path.join(ds, CHROMHMM_DEFAULT)]
                  + [os.path.join(ds, c, "chromhmm_peaks") for c in CALLERS]):
        if os.path.exists(indir):
            for f in os.listdir(indir):
                if (f.endswith(("chr22_binary.txt.gz", "chr22_binary.txt"))
                        and (cell in f or ds in f)):
                    src = os.path.join(indir, f)
                    break
        if src:
            break

    if not src:
        raise MissingInput(f"no chr22 binarization for {ds} chromhmm")

    dst = os.path.join(chr22_indir, os.path.basename(src))
    src_abs = os.path.abspath(src)
    # A link left by an earlier run can point at a moved workdir: broken, so os.path.exists
    # says False, yet os.symlink still fails because the link itself is there. Re-point it.
    if os.path.islink(dst) and os.path.realpath(dst) != src_abs:
        os.unlink(dst)
    if not os.path.lexists(dst):
        os.symlink(src_abs, dst)

    entropies, actual_n_states = [], []
    for n in n_states_range:
        outdir = os.path.join(ds, f"chromhmm_chr22_res_{n}")
        seg_path = os.path.join(outdir, f"{cell}_{n}_segments.bed")
        if not os.path.exists(seg_path):
            cmd = ["java", TOOLS["java_opts"], "-jar",
                   os.path.join(workdir, TOOLS["chromhmm_jar"]),
                   "LearnModel", "-p", "8", "-b", str(CHROMHMM_BIN),
                   chr22_indir, outdir, str(n), P["genome"]]
            print(f"Running ChromHMM for {ds} n={n} ...")
            try:
                subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL,
                               stderr=subprocess.DEVNULL)
            except subprocess.CalledProcessError:
                pass

        if os.path.exists(seg_path):
            segs = analyze.load_bed(seg_path)
            states, counts, state_bp = analyze.build_transition_matrix(segs, CHROMHMM_BIN)
            total_H, _, _, _ = analyze.transition_entropy(states, counts, state_bp)
            entropies.append(total_H)
            actual_n_states.append(len(states))
        else:
            entropies.append(np.nan)
            actual_n_states.append(np.nan)

    return sweep_df(entropy=entropies, actual_n_states=actual_n_states)


def compute_optimal_n_states():
    """Every dataset and method swept over n_states_range, one row per n_states."""
    frames = []
    for ds in DATASETS:
        for sweep, _ in SWEEP_METHODS:
            compute = ((lambda ds=ds: chromhmm_sweep(ds)) if sweep == CHROMHMM
                       else (lambda ds=ds, sweep=sweep: kmeans_sweep(ds, sweep)))
            try:
                df = utils.cached_csv(sweep_cache_path(ds, sweep), compute, sep="\t",
                                      label=f"{ds} {sweep} n_states sweep",
                                      valid=sweep_valid)
            except MissingInput as e:
                print(f"Skipping {ds} {sweep}: {e}")
                continue
            frames.append(df.assign(dataset=ds, method=sweep))
    if not frames:
        return pd.DataFrame(columns=["dataset", "method", "n_states"] + SWEEP_COLUMNS)
    return pd.concat(frames, ignore_index=True)[["dataset", "method", "n_states"] + SWEEP_COLUMNS]


def n_states_valid(df):
    """The combined table is reusable while every method holds a complete sweep."""
    if df.empty or "dataset" not in df.columns or "method" not in df.columns:
        return False
    return all(sweep_valid(group) for _, group in df.groupby(["dataset", "method"]))


df_n_states = utils.cached_csv(n_states_results_path, compute_optimal_n_states,
                               label="optimal n_states results", sep="\t",
                               valid=n_states_valid)
# The method column holds the sweep key (chromhmm | homer | macs2 | omni); the
# plots below label and colour it as the model it sweeps.
df_n_states["Method"] = df_n_states["method"].map(
    lambda s: utils.display_name(SWEEP_METHOD_KEY.get(s, s)))
print(f"{len(df_n_states)} rows over "
      f"{df_n_states.groupby(['dataset', 'method']).ngroups} sweeps")

### 2.2 Plotting

In [ ]:
# 2.2 Plotting: every metric across the datasets of a method (mean ± SE), then
# every dataset on its own per method.
# (column, axis label, title, marker, file part) of the three swept metrics.
SWEEP_PLOTS = [
    ("inertia", "Normalized Inertia", "Elbow Method (Normalized Inertia)", "o", "elbow"),
    ("silhouette", "Silhouette Score", "Silhouette Score", "s", "silhouette"),
    ("entropy", "Transition Matrix Entropy (bits)", "Transition Matrix Entropy", "^", "entropy"),
]

if df_n_states.empty:
    print("  SKIP n_states plots: no sweep on disk")
else:
    sweep_order = [utils.display_name(m) for _, m in SWEEP_METHODS]
    sweep_palette = {utils.display_name(m): utils.method_color(m)
                     for _, m in SWEEP_METHODS}
    shown_order = [m for m in sweep_order if m in set(df_n_states["Method"])]
    n_ds = df_n_states["dataset"].nunique()

    for column, ylabel, title, marker, part in SWEEP_PLOTS:
        data = df_n_states[~df_n_states[column].isna()]
        if data.empty:
            print(f"  SKIP {part}: no {column} in the sweep")
            continue
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.lineplot(data=data, x="n_states", y=column, hue="Method",
                     hue_order=shown_order, palette=sweep_palette,
                     marker=marker, ax=ax)
        ax.set_xlabel("Number of States")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{title} - Combined (n={n_ds})")
        ax.grid(True, linestyle="--", alpha=0.5)
        utils.save_fig(fig, f"out/optimal_n_states_{part}.png")

    # Realized states: a model asked for n does not have to use all of them.
    data = df_n_states[~df_n_states["actual_n_states"].isna()]
    if not data.empty:
        fig, ax = plt.subplots(figsize=(14, 6))
        sns.barplot(data=data, x="n_states", y="actual_n_states", hue="Method",
                    hue_order=shown_order, palette=sweep_palette, ax=ax,
                    capsize=.1, err_kws={"linewidth": 1})
        utils.strip_points(ax, data=data, x="n_states", y="actual_n_states",
                           hue="Method", hue_order=shown_order, size=2)
        ax.set_xlabel("Requested Number of States")
        ax.set_ylabel("Actual Number of States")
        ax.set_title(f"Actual vs Requested Number of States - Combined (n={n_ds})")
        ax.grid(True, linestyle="--", alpha=0.5)
        utils.save_fig(fig, "out/optimal_n_states_actual.png")

    # Per method: one line per dataset, so a dataset that behaves differently
    # from the rest can be picked out of the combined mean above.
    for sweep, _ in SWEEP_METHODS:
        label = sweep_label(sweep)
        method_data = df_n_states[df_n_states["method"] == sweep]
        for column, ylabel, title, marker, part in SWEEP_PLOTS:
            data = method_data[~method_data[column].isna()]
            if data.empty:
                continue
            fig, ax = plt.subplots(figsize=(10, 6))
            for ds, group in data.groupby("dataset"):
                ax.plot(group["n_states"], group[column], marker=marker,
                        label=ds_title(ds))
            ax.set_xlabel("Number of States")
            ax.set_ylabel(ylabel)
            ax.set_title(f"{title} - {label}")
            ax.grid(True, linestyle="--", alpha=0.5)
            ax.legend(fontsize="small", ncol=2)
            utils.save_fig(fig, f"out/optimal_n_states_{part}_{label}.png")

### 2.3 Display

In [ ]:
# 2.3 Display
show_group("Optimal number of states - combined over the datasets",
           [(f"out/optimal_n_states_{part}.png", f"{title} (mean ± SE)")
            for _, _, title, _, part in SWEEP_PLOTS]
           + [("out/optimal_n_states_actual.png",
               "Actual vs requested number of states (mean ± SE)")],
           width=800, level=2)

for _sweep, _ in SWEEP_METHODS:
    _label = sweep_label(_sweep)
    show_group(f"Optimal number of states - {_label}",
               [(f"out/optimal_n_states_{part}_{_label}.png", f"{title} - {_label}")
                for _, _, title, _, part in SWEEP_PLOTS],
               width=800, level=3)

# 3. Cell type differences in chromatin

How often a locus keeps its state across the ChIP-seq datasets, per method: for every
state, the share of the genome that carries it in one dataset, in two, and so on up to
all of them. A method whose states follow the biology of the sample places the active
states differently per cell type and agrees on the background; one whose states follow
its own peak-calling behaviour repeats itself everywhere.

`window=1000` counts a state as present when it is within 1 kb of the locus, `window=0`
requires the locus itself, so the two windows bracket how much of the difference is
positional jitter.

### 3.1 Compute

In [ ]:
# 3.1 Compute: load every method's segmentation of every ChIP-seq dataset, then
# count per state how many datasets share it, per window. The counts are cached
# per method and window - the sweep reads every segmentation of a method at
# once, which is the expensive part.
os.makedirs("out/consistency", exist_ok=True)

CALLER_DISPLAY = {CHROMHMM: CHROMHMM_DISPLAY, HOMER: HOMER_DISPLAY,
                  MACS2: MACS2_DISPLAY, OMNI: OMNI_DISPLAY}
# (label, method key) of every segmentation family compared here: the published
# reference, the individual models under their caller's name, and the joint
# models. The label is what the cache, the plot file and the Display cell below
# all key on, so all three follow one list.
CONSISTENCY_METHODS = (
    [("Reference", "ref")]
    + [(CALLER_DISPLAY[utils.caller_key(m)], m) for m in INTER_DS_METHODS]
    + [(utils.display_name(m), m) for m in JOINT_METHODS])
CONSISTENCY_WINDOWS = [1000, 0]


def consistency_segs(method):
    """Every ChIP-seq segmentation of one method, loaded.

    A joint model covers the two replicates of a dataset rather than the
    dataset as a whole, so it enters through rep1; the reference is the
    published markup of the dataset.
    """
    segs = []
    for ds in CHIP_DATASETS:
        if method == "ref":
            path = ref_bed_path(ds)
        elif method in JOINT_METHODS:
            if ds not in REP_DATASETS:
                continue
            path = inter_ds_bed(f"{ds}/rep1", method)
        else:
            path = inter_ds_bed(ds, method)
        if not os.path.exists(path):
            continue
        try:
            segs.append(match.load_bed(path))
        except Exception as e:
            print(f"  WARNING: could not load {path}: {e}")
    return segs


def consistency_path(label, window, suffix=".pkl"):
    return f"out/consistency/{label.lower()}_w{window}{suffix}"


method_counts = {w: {} for w in CONSISTENCY_WINDOWS}
for label, method in CONSISTENCY_METHODS:
    segs_list = consistency_segs(method)
    if not segs_list:
        print(f"  SKIP {label}: no segmentation on disk")
        continue
    for window in CONSISTENCY_WINDOWS:
        method_counts[window][label] = utils.cached_pickle(
            consistency_path(label, window),
            lambda segs_list=segs_list, window=window:
            analyze.compute_state_consistency(segs_list, bin_size=CHROMHMM_BIN,
                                              window=window, show_progress=True),
            label=f"consistency for {label} (w={window}, {len(segs_list)} datasets)")

### 3.2 Plotting

In [ ]:
# 3.2 Plotting: the state colours of the first ENCODE reference on disk, so the
# states of every method are drawn in the colours the published markup uses.
ref_colors = {}
for ds in CHIP_DATASETS:
    if os.path.exists(ref_bed_path(ds)):
        ref_colors = match.state_colors(match.load_bed(ref_bed_path(ds)))
        break

for window in CONSISTENCY_WINDOWS:
    for label, counts in method_counts[window].items():
        print(f"--- Plotting {label} (w={window}) ---")
        analyze.plot_state_consistency(
            counts, f"{label} (w={window})",
            consistency_path(label, window, suffix="_consistency.png"),
            colors=ref_colors)

### 3.3 Display

In [ ]:
# 3.3 Display
for _window in CONSISTENCY_WINDOWS:
    show_group(f"State consistency across the ChIP-seq datasets (w={_window})",
               [(consistency_path(label, _window, suffix="_consistency.png"),
                 f"{label} (w={_window})")
                for label, _ in CONSISTENCY_METHODS], level=2)

# 4. Replicates: individual vs joint models

`process_encode.sh` fits one model per dataset over both of its replicates — a joint
KMeans per peak caller and a joint ChromHMM over the concatenated binarized signal —
so `rep1` and `rep2` share a single state space, and then relabels both of them to the
ENCODE reference with one shared mapping: the joint state space survives the matching,
and joint and individual segmentations live in the same label space.

This section analyzes every joint segmentation the way section 1 analyzes the
individual ones, compares the two replicates of every method, and answers:

1. **Fragmentation and entropy** — is one model over both replicates simpler than two
   independently learned ones?
2. **Replicate consistency** — do two independently learned models agree across the
   replicates as well as one model learned over both of them, overall and per state?
3. **Individual vs joint** — how much does a replicate's own segmentation change when
   the states are learned jointly?

Both sides of every pair here are matched to the same dataset's ENCODE reference, so
they already share their state names and are compared as they are — no rematching, for
either family, which is also what the per-dataset comparison of section 1.2 does to its
rep1-vs-rep2 pairs (`utils.is_rep_pair`), so that the replicate consistency of a method
is one number wherever it is read off. Kappa and Jaccard come in the two modes
of `MODES`: Full over every state, and NOQH with the Quies/Het bulk of the genome
dropped.

A segmentation that came out truncated or collapsed is dropped before the plots, the way
`summary.ipynb` drops it: the joint ChromHMM of spleen covers 11 Mb of 3.1 Gb in both
replicates, and averaging its 1.8k segments and its agreement into the Joint ChromHMM
bars would read as a simpler, less consistent model rather than as a failed run. Each
bar therefore carries its own n.

### 4.1 Compute

In [ ]:
# 4.1 Compute: analyze every joint segmentation, then compare the individual and
# the joint segmentation of both replicates, per dataset.
JOINT_OUT = f"{SP}/joint"
os.makedirs(JOINT_OUT, exist_ok=True)


def joint_dir(ds):
    """Directory holding the replicate comparison of one dataset."""
    return f"out/{ds}/{MATCH_METHOD}/joint"


def joint_inputs(folder, method):
    """Binarized signal behind a joint segmentation, for its emissions.

    A joint model is learned over both replicates but segments each of them on
    its own, so the emissions of a replicate come from that replicate's signal.
    """
    if method == JOINT_CHROMHMM:
        return [f"{folder}/{CHROMHMM_DEFAULT}/*.txt"]
    caller = method.replace("joint_kmeans_", "")
    return [f"{folder}/{caller}/chromhmm_peaks/chr*.txt.gz"]


def analyze_joint_segmentations():
    """run_analyze per joint segmentation, one per replicate and method."""
    futures = {}
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
        for ds in REP_DATASETS:
            for rep in REPS:
                folder = f"{ds}/{rep}"
                for method in JOINT_METHODS:
                    submit_analyze(executor, futures,
                                   f"{method} ({rep}) for {ds}",
                                   inter_ds_bed(folder, method),
                                   analysis_dir(folder, method), ds,
                                   inputs=joint_inputs(folder, method))
    drain(futures)


def compare_replicate_segmentations():
    """Compare rep1 against rep2 for every individual and joint method.

    all_pairs=False keeps the same-method rep1-vs-rep2 pairs the plots below read;
    the entropy and segment statistics cover every segmentation regardless. The
    stamp next to the results holds the segmentations that produced them, so a
    re-run only recomputes a dataset whose segmentations changed.
    """
    for ds in REP_DATASETS:
        segs = existing([inter_ds_bed(f"{ds}/{rep}", method)
                         for method in REP_METHODS for rep in REPS])
        if len(segs) < 2:
            print(f"  SKIP {ds}: {len(segs)} replicate segmentation(s) on disk")
            continue

        outdir = joint_dir(ds)
        stamp = f"{outdir}/compare_stamp.json"
        signature = {"segs": {p: utils.file_stamp(p) for p in segs}}
        outputs = [f"{outdir}/{name}" for name in
                   ("segment_stats.tsv", "entropy_summary.tsv", "per_state_metrics.tsv")]
        if utils.stamp_current(stamp, signature, outputs):
            print(f"  {ds} replicate comparison is up to date, skipped.")
            continue

        print(f"  {ds}: comparing {len(segs)} replicate segmentations ...")
        compare.run_compare(seg=segs, bins=[seg_bin(p) for p in segs], outdir=outdir)
        utils.save_stamp(stamp, signature)


analyze_joint_segmentations()
compare_replicate_segmentations()

In [ ]:
# 4.1 Compute: the tables the plots read - aggregated per dataset off the
# comparison above, plus the agreement of the segmentation pairs, which is
# computed from the BEDs and therefore cached.
def load_replicate_tsv(filename):
    """Concatenated {joint_dir}/<filename> over the datasets with replicates.

    Splits the segmentation label of every row into its method and replicate.
    """
    frames = [pd.read_csv(f"{joint_dir(ds)}/{filename}", sep="\t").assign(Dataset=ds)
              for ds in REP_DATASETS if os.path.exists(f"{joint_dir(ds)}/{filename}")]
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    keys = df["segmentation"].map(lambda label: label.rsplit("_", 1))
    df["Method"] = keys.map(lambda k: utils.display_name(k[0]))
    df["Rep"] = keys.map(lambda k: k[1])
    return df[df["Method"].isin(REP_METHOD_LABELS)]


def load_replicate_per_state():
    """Per-state rep1-vs-rep2 metrics of every method, over the same datasets."""
    rows = []
    for ds in REP_DATASETS:
        path = f"{joint_dir(ds)}/per_state_metrics.tsv"
        if not os.path.exists(path):
            continue
        for _, row in pd.read_csv(path, sep="\t").iterrows():
            method1, rep1 = str(row["seg1"]).rsplit("_", 1)
            method2, rep2 = str(row["seg2"]).rsplit("_", 1)
            if method1 != method2 or {rep1, rep2} != {"rep1", "rep2"}:
                continue
            rows.append({"Dataset": ds, "Method": utils.display_name(method1),
                         "State": row["state"],
                         KAPPA_DISPLAY: row[KAPPA],
                         JACCARD_DISPLAY: row[JACCARD]})
    return pd.DataFrame(rows)


def compute_joint_rep_agreement():
    """rep1 vs rep2 of every method: two individual models against one joint model."""
    return agreement_table([
        (method, ds,
         inter_ds_bed(f"{ds}/rep1", method), inter_ds_bed(f"{ds}/rep2", method))
        for ds in REP_DATASETS
        for method in REP_METHODS])


def compute_joint_indiv_agreement():
    """Every replicate's own segmentation against its joint counterpart."""
    return agreement_table([
        (indiv, f"{ds}/{rep}",
         inter_ds_bed(f"{ds}/{rep}", indiv), inter_ds_bed(f"{ds}/{rep}", joint))
        for ds in REP_DATASETS
        for rep in REPS
        for indiv, joint in JOINT_INDIV_PAIRS])


# The gate summary.ipynb applies to these same tables, so that a bar here and
# a score there are read off the same segmentations: one covering a fraction of
# what the rest of its dataset covers is a truncated run - the joint ChromHMM
# of spleen covers 11 Mb of 3.1 Gb in both replicates - and one realizing fewer
# than NSTATES states has collapsed, which empties the non-background domain.
# Neither shows up as a missing number, so the segmentation and every
# comparison it enters are dropped rather than averaged into a method's bar.
MIN_COVERAGE_SHARE = 0.5


def unusable_replicates(stats):
    """{(dataset, method, rep)} of the replicate segmentations no plot reads.

    Read off the segment statistics of the section, whose n_segments x
    mean_length is the genome a segmentation covers, against the median of the
    16 segmentations of its own dataset.
    """
    if stats.empty:
        return set()
    covered = stats["n_segments"] * stats["mean_length"]
    share = covered / covered.groupby(stats["Dataset"]).transform("median")
    bad = set()
    for (_, row), sh in zip(stats.iterrows(), share):
        why = []
        if sh < MIN_COVERAGE_SHARE:
            why.append(f"covers {sh:.2%} of the dataset median")
        if row["n_states"] < NSTATES:
            why.append(f"{int(row['n_states'])} of {NSTATES} states")
        if why:
            bad.add((row["Dataset"], row["Method"], row["Rep"]))
            print(f"  EXCLUDE {row['Dataset']} {row['Method']} ({row['Rep']}): "
                  + ", ".join(why))
    return bad


def drop_unusable(df, bad, segs_of, label):
    """*df* without the rows whose segmentations *bad* names.

    *segs_of* maps a row onto the (dataset, method, rep) keys behind it: one
    for a per-segmentation row, both replicates for a rep1-vs-rep2 comparison,
    both models for an individual-vs-joint one, so that a comparison goes when
    either of its sides is unusable.
    """
    if df.empty or not bad:
        return df
    keep = pd.Series([not (set(segs_of(row)) & bad) for _, row in df.iterrows()],
                     index=df.index)
    if not keep.all():
        print(f"  {label}: dropped {int((~keep).sum())} of {len(df)} rows")
    return df[keep]


def _one_seg(row):
    """The single segmentation a per-segmentation row was computed from."""
    return [(row["Dataset"], row["Method"], row["Rep"])]


def _rep_pair(row):
    """Both segmentations a rep1-vs-rep2 row of one method was computed from."""
    return [(row["Dataset"], row["Method"], rep) for rep in REPS]


def _indiv_joint_pair(row):
    """Both models an individual-vs-joint row was computed from.

    Its unit is a replicate folder, "{ds}/{rep}", and its method the individual
    one of the pair, so the joint counterpart has to be named.
    """
    rep = row["Dataset"].split("/")[-1]
    return [(ds_of(row["Dataset"]), method, rep)
            for method in (row["Method"], JOINT_LABEL_OF.get(row["Method"]))]


# The individual method of a (individual, joint) pair to its joint one, in the
# display labels every table of this section is keyed by.
JOINT_LABEL_OF = {utils.display_name(indiv): utils.display_name(joint)
                  for indiv, joint in JOINT_INDIV_PAIRS}

# Read straight off the tables written above, so they follow a re-run.
df_joint_stats = load_replicate_tsv("segment_stats.tsv")
df_joint_entropy = load_replicate_tsv("entropy_summary.tsv")
df_joint_state = load_replicate_per_state()

df_joint_rep = utils.cached_pickle(
    "out/df_joint_rep.pkl", compute_joint_rep_agreement,
    label="replicate consistency, individual and joint", valid=has_modes)
df_joint_indiv = utils.cached_pickle(
    "out/df_joint_indiv.pkl", compute_joint_indiv_agreement,
    label="individual vs joint agreement", valid=has_modes)

# The gate, applied after the caches are written so that each keeps every pair
# it could compute and summary.ipynb goes on applying its own gate to them.
UNUSABLE = unusable_replicates(df_joint_stats)
df_joint_stats = drop_unusable(df_joint_stats, UNUSABLE, _one_seg,
                               "segment stats")
df_joint_entropy = drop_unusable(df_joint_entropy, UNUSABLE, _one_seg, "entropy")
df_joint_state = drop_unusable(df_joint_state, UNUSABLE, _rep_pair, "per-state")
df_joint_rep = drop_unusable(df_joint_rep, UNUSABLE, _rep_pair,
                             "replicate consistency")
df_joint_indiv = drop_unusable(df_joint_indiv, UNUSABLE, _indiv_joint_pair,
                               "individual vs joint")

### 4.2 Plotting

In [ ]:
# 4.2 Plotting: one bar per method, mean ± SE across datasets, every observation
# shown, the joint models hatched. n_per_method throughout: the gate above takes
# an unusable segmentation out of the method it belongs to alone, so the methods
# no longer all cover the same datasets and one n in the title would be wrong
# for the gated bars.
method_barplot(df_joint_stats, "n_segments", REP_METHODS,
               "Number of segments per replicate: individual vs joint models",
               f"{JOINT_OUT}/joint_n_segments.png",
               ylabel="Number of segments", ylim=None, fmt="{:.0f}",
               n_per_method=True)
method_barplot(df_joint_entropy, "total_entropy", REP_METHODS,
               "Transition matrix entropy per replicate: individual vs joint models",
               f"{JOINT_OUT}/joint_entropy.png",
               ylabel="Entropy (bits)", ylim=None, n_per_method=True)

mode_barplots(df_joint_rep, REP_METHODS,
              "Replicate consistency: individual vs joint models",
              JOINT_OUT, "joint_rep_consistency", n_per_method=True)
mode_barplots(df_joint_indiv, INTER_DS_METHODS,
              "Individual vs joint segmentation of a replicate",
              JOINT_OUT, "joint_indiv", n_per_method=True)

# The per-state table has no cosine: a composition vector of a single state has
# one component, so its cosine would be 1.0 by construction.
for _metric, _label in METRICS[:2]:
    state_barplot(df_joint_state, _label, REP_METHODS,
                  f"Per-state replicate consistency ({_label})",
                  f"{JOINT_OUT}/joint_rep_per_state_{_metric}.png",
                  ylabel=f"Replicate {_label}")

### 4.3 Display

The replicate-consistency figures of the individual methods come from the summary
plots of section 1.4; the ones below them compare those individual models against the
joint ones. Both are rep1 vs rep2 of one method, computed the same way, so the two sets
of numbers agree — the summary plots simply cover the individual methods only, since the
per-dataset comparison of section 1.2 does not read the joint models. They also carry no
coverage gate, so the truncated spleen joint ChromHMM the plots below exclude is absent
from them for a different reason: it is not one of their methods.

In [ ]:
# 4.3 Display
show_group("Replicate consistency (summary plots, individual methods)", [
    (f"{SP}/rep_consistency_distribution.png", f"De-novo replicate consistency — {FULL_DISPLAY}"),
    (f"{SP}/rep_consistency_distribution_noqh.png", f"De-novo replicate consistency — {NOQH_DISPLAY}"),
    (f"{SP}/rep_consistency_per_state_jaccard.png", "Per-state replicate consistency: Jaccard"),
    (f"{SP}/rep_consistency_per_state_kappa.png", "Per-state replicate consistency: Cohen's Kappa"),
    (f"{SP}/rep_consistency_per_state_cosine.png", "Per-state replicate consistency: Cosine"),
], level=2)

show_group("Replicate consistency (summary plots, detail)",
           sorted(glob.glob(f"{SP}/rep_consistency_*.png")), level=2)

show_group("Joint models across replicates — segments and entropy", [
    (f"{JOINT_OUT}/joint_n_segments.png", "Number of segments per replicate segmentation"),
    (f"{JOINT_OUT}/joint_entropy.png", "Transition matrix entropy per replicate segmentation"),
], level=2)

show_group("Joint models across replicates — replicate consistency",
           mode_items(JOINT_OUT, "joint_rep_consistency",
                      "two individual models vs one joint model")
           + [(f"{JOINT_OUT}/joint_rep_per_state_{_metric}.png",
               f"Per-state replicate consistency ({_label})")
              for _metric, _label in METRICS[:2]], level=2)

show_group("Joint models across replicates — individual vs joint",
           mode_items(JOINT_OUT, "joint_indiv",
                      "individual vs joint segmentation of the same replicate"),
           level=2)

# Per-segmentation analysis of the joint models, for the datasets DS_DETAIL names.
# for ds in [d for d in DS_DETAIL if d in REP_DATASETS]:
#     for method in JOINT_METHODS:
#         show_group(f"{ds_title(ds)} — {utils.display_name(method)} (rep1)", [
#             (method_plot(f"{ds}/rep1", method, "enrichment/enrichment.png", MATCH_METHOD),
#              "functional enrichment"),
#             (method_plot(f"{ds}/rep1", method, "bin_emissions/state_emissions.png", MATCH_METHOD),
#              "binarized emissions"),
#             (method_plot(f"{ds}/rep1", method, "segment_length.png", MATCH_METHOD),
#              "segment length distribution"),
#         ], width=760, level=4)

# 5. Sample-pair agreement: across samples and across assays

The same method run on two different samples, over **every pair of datasets** and for
**every de-novo method** — the four individual models and the four joint ones. It
answers how much of a segmentation is the biology of the sample rather than the
method's own signature: a method whose states are driven by its own peak-calling
behaviour rather than by the data reproduces itself across cell types and scores high
here, so — unlike the replicate agreement of section 4 — a *higher* number is not
automatically better; it is read next to the replicate agreement, which the same
method should score much higher on.

Both views come from **one** sweep over the dataset pairs, split by what changes
between the two samples:

- **Across samples** — the same-assay pairs: ChIP-seq against ChIP-seq, Mint-ChIP
  against Mint-ChIP. The bars of `out/summary_plots/cross_sample`.
- **Across assays** — the ChIP-seq × Mint-ChIP pairs, where the assay changes as well
  as the sample. The bars of `out/summary_plots/cross_assay`. Only
  monocytes / monocytes_mint share a biosample, so this axis measures a change of
  assay and of cell type together over all of its pairs alike; the run-time summary
  prints the same-biosample pair on its own, so the cost of the cross-cell-type pairs
  can be read off.

Each method is swept over every pair of the datasets it actually segmented, so a
method that covers fewer datasets contributes fewer pairs instead of cutting the
comparison down for all of them. The individual models cover every dataset and are
read off their pooled segmentation. The joint models cover fewer: `process_encode.sh`
fits one joint model per dataset over that dataset's two replicates — there is no
cohort-wide ENCODE joint model the way the 1000-epigenomes pipeline has one over all
epigenomes — so a joint model exists only for the datasets that have replicates, and
is read off `rep1`.

The two sides of a pair were matched to two *different* ENCODE reference markups, so
their state names agree only as far as the two markups do, and a cross-assay pair not
at all: the ChIP markups name their states Enh1 / Enh2 / EnhG1 / EnhG2 / TssFlnkD and
the Mint ones Enh / EnhLo / EnhG / ReprPCWk / Unknown, so 5 of the 15 names exist on
one side only and just 7 non-background names are shared. Every pair is therefore
realigned by maximum overlap before the metrics are read
(`pair_agreement(..., rematch=True)`), the way section 1.3 runs
`run_compare(..., rematch=True)`, so a caller is not charged for the two markups
disagreeing about which state a name belongs to.

### 5.1 Compute

In [ ]:
# 5.1 Compute: agreement between two samples segmented by the same method, over
# every pair of datasets, for every de-novo method. One sweep, both views: the
# same-assay pairs are the cross-sample axis and the ChIP x Mint ones the
# cross-assay axis, so no pair is compared twice.
CS_OUT = f"{SP}/cross_sample"
XA_OUT = f"{SP}/cross_assay"
os.makedirs(CS_OUT, exist_ok=True)
os.makedirs(XA_OUT, exist_ok=True)


def sample_pair_folders(method):
    """{dataset: folder} a method's cross-sample segmentation is read from.

    An individual model is fitted per dataset, so it covers every dataset and
    is read off the pooled segmentation. A joint model is fitted per dataset
    over that dataset's two replicates, so it exists only for the datasets that
    have replicates, and is read off rep1.
    """
    if method in JOINT_METHODS:
        return {ds: f"{ds}/rep1" for ds in REP_DATASETS}
    return {ds: ds for ds in DATASETS}


def pair_order_key(ds):
    """Sort key of a dataset within a pair: the Mint-ChIP side goes second.

    pair_agreement(rematch=True) realigns the state space of its *second*
    argument onto its first, and the two vocabularies are not equals: the ChIP
    markups carry the full 15 state names and the Mint ones name 5 states of
    their own. Ordering the ChIP side first therefore maps the Mint vocabulary
    onto the ChIP one, which is what the cross-assay view of the sweep reads.
    Same-assay pairs keep the order the datasets have in the configuration.
    """
    return (ds.endswith("_mint"), list(DATASETS).index(ds))


def sample_pair_tasks():
    """(method, ds_a, ds_b, path_a, path_b) of every pair a method can reach."""
    tasks = []
    for method in REP_METHODS:
        folders = sample_pair_folders(method)
        have = {ds: inter_ds_bed(folder, method)
                for ds, folder in folders.items()}
        have = {ds: path for ds, path in have.items() if os.path.exists(path)}
        missing = [ds for ds in folders if ds not in have]
        if missing:
            print(f"  {method}: no segmentation for {', '.join(missing)}")
        for ds_a, ds_b in combinations(sorted(have, key=pair_order_key), 2):
            tasks.append((method, ds_a, ds_b, have[ds_a], have[ds_b]))
    return tasks


def sample_pair_worker(task):
    """One pair of the sweep; runs in a worker process, which loads its own two
    segmentations - a dense ENCODE segmentation and its index run to hundreds of
    MB, so holding every dataset of a method in the parent would not fit."""
    return pair_agreement(task[3], task[4], rematch=True)


def compute_sample_pair_agreement():
    """Every pair of samples a method reaches, per de-novo method.

    One row per pair and mode, the shape agreement_table() writes, so a plot
    and summary.ipynb both select their domain by filtering on Mode.
    """
    tasks = sample_pair_tasks()
    if not tasks:
        return pd.DataFrame()
    print(f"  {len(tasks)} pairs over {len({t[0] for t in tasks})} methods ...")
    # Each worker holds two segmentations, so the pool is capped rather than
    # taking every core.
    with ProcessPoolExecutor(max_workers=min(8, os.cpu_count()),
                             mp_context=mp.get_context("fork")) as executor:
        results = list(executor.map(sample_pair_worker, tasks, chunksize=1))

    rows = []
    for (method, ds_a, ds_b, _, _), modes in zip(tasks, results):
        if modes is None:
            print(f"  SKIP {method}: {ds_a} vs {ds_b}")
            continue
        # A metric of 0.0 in the NOQH domain is a result, not a gap: it is what
        # a segmentation that collapsed its states scores once the background is
        # dropped, and n_states.png already reports which ones those are.
        rows += agreement_row(
            method, f"{ds_a}/{ds_b}", modes,
            Joint=method in JOINT_METHODS,
            # A ChIP-seq sample against a Mint-ChIP one changes the assay as
            # well as the sample; the two are plotted as separate axes below.
            SameAssay=ds_a.endswith("_mint") == ds_b.endswith("_mint"))
    return pd.DataFrame(rows)


def canonical_pairs(df):
    """Cache guard: every mixed pair holds its ChIP-seq side first.

    A sweep from before pair_order_key() wrote some of them the other way
    round, which realigns the ChIP vocabulary onto the Mint one instead - the
    metrics of such a pair are not the ones the cross-assay view reports, so
    the cache is rebuilt rather than reused.
    """
    sides = [str(p).split("/") for p in df["Dataset"]]
    return not any(a.endswith("_mint") and not b.endswith("_mint")
                   for a, b in sides)


def same_biosample(pair):
    """True when the two sides of a pair are the same biosample, one of them
    assayed with Mint-ChIP: "monocytes/monocytes_mint"."""
    a, b = pair.split("/")
    return f"{a}_mint" == b or f"{b}_mint" == a


df_sample_pairs = utils.cached_pickle(
    "out/df_cross_sample.pkl", compute_sample_pair_agreement,
    label="sample-pair agreement, individual and joint",
    # Rejects a cache from before the SameAssay column, which is the one
    # written before the sweep covered every dataset, and one written before
    # the pairs were ordered by assay.
    valid=lambda df: has_modes(df, ("SameAssay", "Joint")) and canonical_pairs(df))

# The two views of the sweep. df_cross_assay.pkl is written for summary.ipynb,
# which reads the cross-assay axis off its own cache; it is a view of the sweep
# above, not a second comparison of the same pairs.
df_cross_sample = df_sample_pairs[df_sample_pairs["SameAssay"]]
df_cross_assay = (df_sample_pairs[~df_sample_pairs["SameAssay"]
                                  & ~df_sample_pairs["Joint"]]
                  .drop(columns=["SameAssay", "Joint"]))
df_cross_assay = df_cross_assay.assign(
    SameBiosample=df_cross_assay["Dataset"].map(same_biosample))
df_cross_assay.to_pickle("out/df_cross_assay.pkl")
print(f"  saved out/df_cross_assay.pkl "
      f"({df_cross_assay['Dataset'].nunique()} ChIP x Mint-ChIP pairs)")

mean_agreement(df_cross_sample, "same-assay pairs")
mean_agreement(df_cross_assay, "ChIP x Mint-ChIP pairs")
mean_agreement(df_cross_assay, "same-biosample pairs",
               mask=df_cross_assay["SameBiosample"])

### 5.2 Plotting

In [ ]:
# 5.2 Plotting: one bar per de-novo method, mean ± SE over the dataset pairs,
# every pair shown, the joint models hatched.
#
# No bar here compares a method against another one, every bar is one method
# against itself on two samples - this is not the individual-vs-joint
# comparison of section 4.
#
# Across samples: the individual models carry every pair of the datasets, the
# joint ones only the pairs of the datasets that have replicates, so the bars
# sit on different numbers of observations and each carries its own n on its
# label.
mode_barplots(df_cross_sample, REP_METHODS,
              "Cross-sample agreement over dataset pairs",
              CS_OUT, "cross_sample",
              point_size=2, point_alpha=0.35, n_per_method=True)

# Across assays: the joint models have no cross-assay counterpart - one joint
# model covers the two replicates of a single dataset and no Mint-ChIP dataset
# has replicates - so this is the four individual methods only.
mode_barplots(df_cross_assay, INTER_DS_METHODS,
              "ChIP-seq vs Mint-ChIP agreement",
              XA_OUT, "cross_assay")

### 5.3 Display

The distribution plots come from the name-matched similarity matrices of section 1.3
and cover the individual models; the bars come from the sweep of 5.1, which realigns
the state spaces and covers the joint models as well.

In [ ]:
# 5.3 Display
show_group("Inter-dataset similarity (summary plots)", [
    (f"{SP}/method_similarity_distribution.png",
     f"De-novo inter-dataset similarity — {FULL_DISPLAY}"),
    (f"{SP}/method_similarity_distribution_noqh.png",
     f"De-novo inter-dataset similarity — {NOQH_DISPLAY}"),
    (f"{REF}/similarity_distribution.png",
     f"Inter-reference similarity — {FULL_DISPLAY}"),
    (f"{REF}/similarity_distribution_noqh.png",
     f"Inter-reference similarity — {NOQH_DISPLAY}"),
], level=2)

show_group("Across samples (same assay)",
           mode_items(CS_OUT, "cross_sample", "sample vs sample"), level=2)

show_group("Across assays (ChIP-seq vs Mint-ChIP)",
           [(f"{SP}/method_similarity_distribution_chip_vs_mint.png",
             f"Inter-dataset similarity, ChIP-seq vs Mint-ChIP — {FULL_DISPLAY}"),
            (f"{SP}/method_similarity_distribution_chip_vs_mint_noqh.png",
             f"Inter-dataset similarity, ChIP-seq vs Mint-ChIP — {NOQH_DISPLAY}")]
           + mode_items(XA_OUT, "cross_assay", "ChIP-seq vs Mint-ChIP"), level=2)